In [1]:
import json
from pathlib import Path
from typing import List

from langchain_community.document_loaders import TextLoader
from langchain_community.graphs import Neo4jGraph
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import TokenTextSplitter, RecursiveCharacterTextSplitter
from neo4j.exceptions import ClientError

## Init our tools - LLM and Graph database

In [17]:
graph = Neo4jGraph()

# Embeddings & LLM models
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
embedding_dimension = 1536
llm = ChatOpenAI(model="gpt-4o-2024-08-06", temperature=0)

In [18]:
graph.query("match (n:CategorySection|Category|Section|MenuItem|Ingredient|Question|Answer) detach delete n")

[]

## Get the text from PDF file

In [19]:
import re
import pymupdf4llm
from langchain_core.documents.base import Document

def docs_to_json_pretty(docs: List[Document]):
    return json.dumps([doc.dict() for doc in docs], indent=4)

# get the markdown text from pdf
md_text = pymupdf4llm.to_markdown("menu.pdf")

# now work with the markdown text, e.g. store as a UTF8-encoded file
import pathlib
pathlib.Path("dinner.md").write_bytes(md_text.encode())


Processing menu.pdf...
[                                        ] (0/6=====[======                                  ] (1/6======[=============                           ] (2/======[====================                    ] (3/6=====[==========================              ] (4/6======[=================================       ] (5/======[========================================] (6/6]


13610

In [20]:
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("###", "Category"),
]

markdown_splitter = MarkdownHeaderTextSplitter(headers_to_split_on)
md_header3_splits = markdown_splitter.split_text(md_text)
categories = [split.metadata['Category'] for split in md_header3_splits if 'Category' in split.metadata]
categories_str = ', '.join(categories)
categories_str

'APPETIZERS, SOUP & SALAD, PASTA, ENTRÉES, SIDES, PIZZAS, PERSONALIZED PIES'

## Prompt Plumbing for MenuItems and Ingredients

In [21]:
from decimal import Decimal
from typing import List, Dict

class Ingredient(BaseModel):
    """A menu item ingredient."""
    name: str = Field(..., description="Name of this ingredient in lower case letters without any other ingredient names in its name. ")


class MenuItem(BaseModel):
    """A menu item and its list of ingredients."""
    title: str = Field(..., description="Title for this menu item")
    # price: Decimal = Field(..., description="Price for this menu item")
    ingredients: List[Ingredient] = Field(..., description="List of ingredients for this menu item")
    def ingredients_to_list_of_dicts(self) -> List[Dict[str, str]]:
        """Converts the list of ingredients to a list of dictionaries."""
        return [ingredient.dict() for ingredient in self.ingredients]

class MenuItems(BaseModel):
    """Generated menu items out of menu text."""
    items: List[MenuItem] = Field(..., description="List of menu items for a given menu section")

questions_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                f"You are generating menu items with corresponding ingredients "
                "based on the information found in the text of a restaurant menu section. "
                "Each menu item should have a list of ingredients. "
                "Each ingredient name should be captured in lower case letters and cannot contain other ingredients as part of the name. "
            ),
        ),
        (
            "human",
            (
                "Use the given format to generate menu items with corresponding ingredients "
                "from the following input: {input}"
            ),
        ),
    ]
)

menu_items_chain = questions_prompt | llm.with_structured_output(MenuItems)



## Generate and Ingest SectionCategories with MenuItems and Ingredients

In [22]:
for section_id, section in enumerate(md_header3_splits):
    # ignore document sections that do not have a category
    if "Category" not in section.metadata:
        continue

    menu_items_data = menu_items_chain.invoke(section.page_content)
    items = menu_items_data.items
    # print(docs_to_json_pretty(items))

    params = {
        "section_id": f"s-{section_id}",
        "section_title": section.metadata["Category"],
        "menu_items": [
            {
                "item_id": f"s-{section_id}-mi-{item_id}",
                "title": item.title,
                "ingredients": item.ingredients_to_list_of_dicts(),
                # "price": item.price,
                # "item_embedding": embeddings.embed_query(item.title)
            }
            for item_id, item in enumerate(items)
        ],
    }

    # print(json.dumps(params, indent=4))

    # create menu items and ingredient nodes
    graph.query(
        """
        UNWIND $menu_items AS item
        MERGE (cat:CategorySection {id: $section_id, title: $section_title})
        WITH cat, item
        MERGE (mi:MenuItem {id: item.item_id, title: item.title})
        MERGE (mi)-[:IN_SECTION]->(cat)
        WITH mi, item
        UNWIND item.ingredients as ingredient
        MERGE (ing:Ingredient {text: ingredient.name})
        MERGE (mi)-[:USES_INGREDIENT]->(ing)
    """,
        params,
    )


## Prompt Plumbing for Jeopardy Questions and Answers

In [23]:
class JeopardyQuestion(BaseModel):
    """Structuring questions with categories, points, and answers."""
    category: str = Field(..., description="Jeopardy-style category")
    question: str = Field(..., description="Generated question")
    answer: str = Field(..., description="Generated answer")
    points: int = Field(..., description="Point value associated with the question")

class JeopardyQuestions(BaseModel):
    """Generating hypothetical Jeopardy-style questions with answers."""
    questions: List[JeopardyQuestion] = Field(
        ..., description="List of questions with their categories, answers, and points"
    )

questions_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            (
                f"You are generating Jeopardy-style questions with points and answers in the following categories {categories_str}"
                "based on the information found in the text of a restaurant menu. "
                "Do not generate questions about the cost of a menu item. "
                "Each question should have only one correct answer and a corresponding category. "
                "Each question should have a point value of 100 or 200 or 300 or 400 or 500 depending on the difficulty of the question. "
            ),
        ),
        (
            "human",
            (
                "Use the given format to generate Jeopardy-style questions with difficulty point value and answers in corresponding category from the "
                "following input: {input}"
            ),
        ),
    ]
)

question_chain = questions_prompt | llm.with_structured_output(JeopardyQuestions)



## Generate and Ingest Questions with Points and Answers for Categories

In [24]:
for section_id, section in enumerate(md_header3_splits):
    # ignore document sections that do not have a category
    if "Category" not in section.metadata:
        continue

    jeopardy_data = question_chain.invoke(section.page_content)
    questions = jeopardy_data.questions
    # print(docs_to_json_pretty(questions))

    params = {
        "section_id": f"s-{section_id}",
        "section_title": section.metadata["Category"],
        "questions": [
            {
                "question_id": f"s-{section_id}-q-{question_id}",
                "text": q.question,
                "points": q.points,
                "question_embedding": embeddings.embed_query(q.question),
                "answer_id": f"c-{section_id}-a-{question_id}",
                "answer_text": q.answer,
                "answer_embedding": embeddings.embed_query(q.answer)
            }
            for question_id, q in enumerate(questions)
        ],
    }

    # print(json.dumps(params, indent=4))

        # create question nodes
    graph.query(
        """
        UNWIND $questions AS question
        MERGE (cat:CategorySection {id: $section_id, title: $section_title})
        WITH cat, question
        MERGE (q:Question {id: question.question_id, text: question.text, points: question.points})
        MERGE (a:Answer {text: question.answer_text})
        MERGE (a)<-[:HAS_ANSWER]-(q)-[:IN_CATEGORY]->(cat)
    """,
        params,
    )


In [25]:
from langchain_community.vectorstores import Neo4jVector

def create_index(index_name, node_label):
    neo4j_vector = Neo4jVector.from_existing_graph(
        embedding=embeddings,
        index_name = index_name,
        node_label = node_label,
        text_node_properties=["text"],
        embedding_node_property="embedding",
        retrieval_query="RETURN node.id, node.text AS text, score, {} AS metadata"
    )
    return neo4j_vector

graph.query("drop index ingredients_index if exists")
graph.query("drop index questions_index if exists")
graph.query("drop index answers_index if exists")

ingredients_index = create_index("ingredients_index", "Ingredient")
questions_index = create_index("questions_index", "Question")
answers_index = create_index("ingredients_index", "Answer")

Failed to write data to connection IPv4Address(('127.0.0.1', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687)))
Failed to write data to connection IPv4Address(('127.0.0.1', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687)))
Failed to write data to connection IPv4Address(('127.0.0.1', 7687)) (ResolvedIPv4Address(('127.0.0.1', 7687)))


In [26]:
sim_search_response = questions_index.similarity_search("garlic")
print(docs_to_json_pretty(sim_search_response))

[
    {
        "id": null,
        "metadata": {},
        "page_content": "This side dish features a green vegetable saut\u00e9ed with garlic and olive oil.",
        "type": "Document"
    },
    {
        "id": null,
        "metadata": {},
        "page_content": "This side dish is broiled and flavored with lemon and garlic.",
        "type": "Document"
    },
    {
        "id": null,
        "metadata": {},
        "page_content": "This side dish includes broccolini saut\u00e9ed with chopped tomato and garlic.",
        "type": "Document"
    },
    {
        "id": null,
        "metadata": {},
        "page_content": "Which entr\u00e9e is served with roasted garlic mashed potatoes and roasted asparagus?",
        "type": "Document"
    }
]


## Generate PageRank for MenuItem Ingredients

In [27]:
graph.query("""
CALL gds.graph.drop('menu-item-ingredients')
""")
            
graph.query("""
MATCH (source:MenuItem)-[:USES_INGREDIENT]->(target:Ingredient)
WITH gds.graph.project('menu-item-ingredients', source, target) AS g
RETURN g.graphName as graph, g.nodeCounts as nodes, g.relationshipCounts as rels            
""")

graph.query("""
CALL gds.pageRank.write('menu-item-ingredients', {writeProperty: 'pageRank'})
""")


[{'writeMillis': 17,
  'nodePropertiesWritten': 190,
  'ranIterations': 2,
  'didConverge': True,
  'centralityDistribution': {'min': 0.14999961853027344,
   'max': 0.43687629699707026,
   'p90': 0.27466678619384766,
   'p999': 0.4368753433227539,
   'p99': 0.4049997329711914,
   'p50': 0.1712493896484375,
   'p75': 0.19249916076660156,
   'p95': 0.30300045013427734,
   'mean': 0.18757852253160978},
  'postProcessingMillis': 6,
  'preProcessingMillis': 0,
  'computeMillis': 10,
  'configuration': {'writeProperty': 'pageRank',
   'jobId': '4c5e0f7f-f706-434f-a2a2-0bacc0275799',
   'scaler': 'NONE',
   'sourceNodes': [],
   'sudo': False,
   'maxIterations': 20,
   'dampingFactor': 0.85,
   'writeToResultStore': False,
   'writeConcurrency': 4,
   'logProgress': True,
   'nodeLabels': ['*'],
   'concurrency': 4,
   'relationshipTypes': ['*'],
   'tolerance': 1e-07}}]